# Distributed Ray Tasks & Actors from Kubeflow Workspace on GKE

This Jupyter Notebook demonstrates using a **Kubeflow Workspace** as the unified entry point to your Kubernetes (GKE) cluster to:
1. **Dynamically create a RayCluster directly from Python** inside this notebook using the in-cluster `default-editor` ServiceAccount.
2. **Wait for the RayCluster** head and worker nodes to be provisioned and ready.
3. **Connect to the RayCluster** via the Ray Client (`ray://kubeflow-raycluster-head-svc:10001`).
4. **Execute distributed Ray tasks and stateful Ray actors** across worker nodes.
5. **Clean up and delete the RayCluster** dynamically from Python when finished.

## 1. Create the RayCluster Directly from this Notebook

In [ ]:
import time
import yaml
from kubernetes import client, config

# Load in-cluster Kubernetes configuration (mounted at /var/run/secrets/kubernetes.io/serviceaccount)
config.load_incluster_config()
custom_api = client.CustomObjectsApi()
core_v1 = client.CoreV1Api()

namespace = "kubeflow-user-example-com"
cluster_name = "kubeflow-raycluster"

raycluster_yaml = """
apiVersion: ray.io/v1
kind: RayCluster
metadata:
  name: kubeflow-raycluster
  namespace: kubeflow-user-example-com
spec:
  rayVersion: '2.56.1'
  headGroupSpec:
    rayStartParams:
      num-cpus: '1'
      dashboard-host: '0.0.0.0'
    template:
      metadata:
        labels:
          sidecar.istio.io/inject: "false"
      spec:
        serviceAccountName: default-editor
        containers:
        - name: ray-head
          image: rayproject/ray:2.56.1-py311-cpu
          resources:
            limits:
              cpu: "1"
              memory: "2G"
            requests:
              cpu: "200m"
              memory: "512Mi"
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
            runAsNonRoot: true
            seccompProfile:
              type: RuntimeDefault
  workerGroupSpecs:
    - replicas: 2
      minReplicas: 1
      maxReplicas: 4
      groupName: worker-group
      rayStartParams:
        num-cpus: '1'
      template:
        metadata:
          labels:
            sidecar.istio.io/inject: "false"
        spec:
          serviceAccountName: default-editor
          containers:
          - name: ray-worker
            image: rayproject/ray:2.56.1-py311-cpu
            resources:
              limits:
                cpu: "1"
                memory: "1G"
              requests:
                cpu: "200m"
                memory: "512Mi"
            securityContext:
              allowPrivilegeEscalation: false
              capabilities:
                drop: ["ALL"]
              runAsNonRoot: true
              seccompProfile:
                type: RuntimeDefault
"""

manifest = yaml.safe_load(raycluster_yaml)

try:
    custom_api.create_namespaced_custom_object(
        group="ray.io",
        version="v1",
        namespace=namespace,
        plural="rayclusters",
        body=manifest
    )
    print(f"✅ RayCluster '{cluster_name}' created successfully from notebook!")
except client.exceptions.ApiException as e:
    if e.status == 409:
        print(f"ℹ️ RayCluster '{cluster_name}' already exists.")
    else:
        raise e


## 2. Wait for RayCluster Pods to be Running

In [ ]:
print("Waiting for RayCluster pods to be running...")
for i in range(30):
    pods = core_v1.list_namespaced_pod(
        namespace=namespace,
        label_selector=f"ray.io/cluster={cluster_name}"
    )
    running_pods = [p.metadata.name for p in pods.items if p.status.phase == "Running"]
    pod_str = ", ".join(running_pods) if running_pods else "waiting..."
    print(f"  Attempt {i+1}/30: {len(running_pods)}/{len(pods.items)} Ray pods running ({pod_str})")
    if len(running_pods) >= 3:
        print("\n✅ All RayCluster pods (1 Head + 2 Workers) are Running!")
        break
    time.sleep(4)


## 3. Connect to the GKE RayCluster via Ray Client

In [ ]:
import ray
import os
import socket

# Connect to the RayCluster Head Service running in the user namespace
ray_head_address = "ray://kubeflow-raycluster-head-svc:10001"

print(f"Connecting to RayCluster at {ray_head_address}...")
ray.init(address=ray_head_address)

resources = ray.cluster_resources()
cpus = resources.get('CPU', 0)
memory_gb = resources.get('memory', 0) / (1024**3)

print("\n✅ Successfully connected to RayCluster!")
print(f"Available Cluster CPUs: {cpus}")
print(f"Available Cluster Memory: {memory_gb:.2f} GB")


## 4. Launch Distributed Remote Tasks Across Ray Nodes

In [ ]:
@ray.remote
def compute_distributed_task(task_id):
    import time
    time.sleep(0.3)  # Simulate compute workload
    hostname = socket.gethostname()
    pid = os.getpid()
    result = task_id ** 3
    return {
        "task_id": task_id,
        "result": result,
        "node": hostname,
        "pid": pid
    }

print("Submitting 8 parallel distributed tasks to RayCluster...")
futures = [compute_distributed_task.remote(i) for i in range(8)]
task_results = ray.get(futures)

print("\nDistributed Tasks Results:")
nodes_used = set()
for res in task_results:
    nodes_used.add(res['node'])
    print(f"  -> Task {res['task_id']} (Result: {res['result']}) computed on Node: {res['node']} (PID: {res['pid']})")

print(f"\n🚀 Tasks executed across {len(nodes_used)} distinct Ray nodes: {nodes_used}")


## 5. Launch Distributed Stateful Ray Actors Across Ray Nodes

In [ ]:
@ray.remote
class DistributedStateCounter:
    def __init__(self, actor_name):
        self.name = actor_name
        self.counter = 0
        self.hostname = socket.gethostname()

    def increment(self, step=1):
        self.counter += step
        return self.counter

    def get_info(self):
        return {"actor": self.name, "count": self.counter, "node": self.hostname}

print("Creating 4 stateful distributed actors across the cluster...")
actors = [DistributedStateCounter.remote(f"worker_actor_{i}") for i in range(4)]

# Increment state on all actors concurrently
ray.get([actor.increment.remote(10) for actor in actors])
actor_states = ray.get([actor.get_info.remote() for actor in actors])

print("\nStateful Actor Cluster States:")
for state in actor_states:
    print(f"  -> Actor [{state['actor']}] on Node [{state['node']}] -> Count = {state['count']}")


## 6. Teardown & Clean Up RayCluster Directly from Notebook

In [ ]:
# Disconnect Ray Client
ray.shutdown()

print(f"Deleting RayCluster '{cluster_name}' via Kubernetes Python API...")
try:
    custom_api.delete_namespaced_custom_object(
        group="ray.io",
        version="v1",
        namespace=namespace,
        plural="rayclusters",
        name=cluster_name
    )
    print(f"✅ RayCluster '{cluster_name}' deletion requested.")
except client.exceptions.ApiException as e:
    if e.status == 404:
        print(f"ℹ️ RayCluster '{cluster_name}' already deleted.")
    else:
        raise e

print("\nWaiting for RayCluster pods to terminate cleanly...")
for i in range(30):
    pods = core_v1.list_namespaced_pod(
        namespace=namespace,
        label_selector=f"ray.io/cluster={cluster_name}"
    )
    if len(pods.items) == 0:
        print(f"✅ All RayCluster '{cluster_name}' pods terminated successfully!")
        break
    print(f"  Attempt {i+1}/30: {len(pods.items)} pods remaining...")
    time.sleep(3)
